In [1]:
!git clone https://github.com/MohammedMosuily/NLP_Lab6.git

Cloning into 'NLP_Lab6'...
remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 5 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (5/5), 30.50 KiB | 10.17 MiB/s, done.


In [2]:
%ls NLP_Lab6/

dataset/  Lab6_deep_learning_for_nlp-3.ipynb


In [3]:
import json

with open('NLP_Lab6/Lab6_deep_learning_for_nlp-3.ipynb', 'r', encoding='utf-8') as f:
    nb = json.load(f)

for i, cell in enumerate(nb.get('cells', [])):
    cell_type = cell.get('cell_type')
    source = ''.join(cell.get('source', []))
    print(f"--- Cell {i} ({cell_type}) ---")
    print(source[:500] + ('\n... [truncated]' if len(source) > 500 else ''))
    print('\n')

--- Cell 0 (markdown) ---
# Deep Learning for NLP

**Deep Learning** is a type of Machine Learning that uses **Neural Networks with multiple layers** to learn patterns from data.

Deep Learning can be used in NLP tasks such as:
- Text Classification
- Sentiment Analysis
- Question Answering

In this lab, we will build a **deep learning model** to classify Yelp restaurant reviews as:

- **0 = Negative**
- **1 = Positive**


--- Cell 1 (markdown) ---
<img src="https://drive.google.com/uc?export=view&id=1rUaLO1wEOcUZ10Kwd0nUvS3jQzrqoOcI" width="550">
<img src="https://drive.google.com/uc?export=view&id=1XKPTKwxmIpP5cZIkDajlai6aI9jUX-C8" width="400">


--- Cell 2 (markdown) ---
## 1. Load the Yelp Dataset

Each row contains:

- a restaurant review
- its sentiment label



--- Cell 3 (code) ---
import pandas as pd

df = pd.read_csv("07-yelp-dataset.txt", sep="\t", names=["review", "label"])

print("Number of reviews:", len(df))
df.head()


--- Cell 4 (code) ---
print(df["label"].value_count

In [4]:
!cp NLP_Lab6/dataset/07-yelp-dataset.txt .
!ls

07-yelp-dataset.txt  NLP_Lab6  sample_data


In [7]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 46.4 MB/s eta 0:00:00


In [8]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
import torch
import numpy as np
from gensim.models import Word2Vec

# 1. Load
df = pd.read_csv("07-yelp-dataset.txt", sep="\t", names=["review", "label"])

# 2. Preprocess
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df["review"] = df["review"].apply(preprocess_text)

# 3. Split
X = df["review"]
y = df["label"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

# Task 1 – Represent Reviews Using Word2Vec
# Tokenize reviews
train_tokens = [review.split() for review in X_train]
test_tokens = [review.split() for review in X_test]

# Train Word2Vec
w2v_model = Word2Vec(sentences=train_tokens, vector_size=200, window=5, min_count=1, sg=1, epochs=10, workers=4)

# Convert review (list of tokens) to a single vector (average word vector)
def get_review_vector(tokens, model, vector_size):
    valid_vectors = [model.wv[word] for word in tokens if word in model.wv]
    if len(valid_vectors) > 0:
        return np.mean(valid_vectors, axis=0)
    else:
        return np.zeros(vector_size)

X_train_w2v = np.array([get_review_vector(tokens, w2v_model, 200) for tokens in train_tokens])
X_test_w2v = np.array([get_review_vector(tokens, w2v_model, 200) for tokens in test_tokens])

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train_w2v, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_w2v, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)

print("Task 1 Complete!")
print("Train tensor shape:", X_train_tensor.shape)
print("Test tensor shape:", X_test_tensor.shape)

Task 1 Complete!
Train tensor shape: torch.Size([800, 200])
Test tensor shape: torch.Size([200, 200])


In [9]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score

# Task 2 – Build a Different Neural Network
class CustomNN(nn.Module):
    def __init__(self, input_dim):
        super(CustomNN, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)

# Initialize model, loss, optimizer
model = CustomNN(input_dim=200)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Create Dataloader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Train the model
epochs = 15
model.train()
for epoch in range(epochs):
    epoch_loss = 0.0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * batch_X.size(0)
    epoch_loss /= len(train_loader.dataset)
    print(f"Epoch {epoch+1:02d}/{epochs:02d} | Train Loss: {epoch_loss:.4f}")

# Evaluate accuracy
model.eval()
with torch.no_grad():
    test_predictions = model(X_test_tensor)
    probs = torch.sigmoid(test_predictions)
    preds = (probs >= 0.5).float()

    accuracy = accuracy_score(y_test_tensor.numpy(), preds.numpy())
    print(f"\nTest Accuracy: {accuracy * 100:.2f}%")

Epoch 01/15 | Train Loss: 0.6943
Epoch 02/15 | Train Loss: 0.6933
Epoch 03/15 | Train Loss: 0.6934
Epoch 04/15 | Train Loss: 0.6941
Epoch 05/15 | Train Loss: 0.6932
Epoch 06/15 | Train Loss: 0.6932
Epoch 07/15 | Train Loss: 0.6933
Epoch 08/15 | Train Loss: 0.6932
Epoch 09/15 | Train Loss: 0.6932
Epoch 10/15 | Train Loss: 0.6932
Epoch 11/15 | Train Loss: 0.6932
Epoch 12/15 | Train Loss: 0.6935
Epoch 13/15 | Train Loss: 0.6932
Epoch 14/15 | Train Loss: 0.6932
Epoch 15/15 | Train Loss: 0.6935

Test Accuracy: 50.00%
